In [1]:
# ============================================
# === Cell 1: Imports & Global Config     ===
# ============================================

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    average_precision_score,
    mean_squared_error,
)

# Try importing LightGBM
try:
    import lightgbm as lgb
except ImportError as e:
    raise ImportError(
        "lightgbm is required. Install it with `pip install lightgbm`."
    ) from e

# Global config
DATA_PATH = Path("cimis_all_stations_clean.csv")   # CSV must be in same folder as notebook
COOL_MONTHS = [10, 11, 12, 1, 2, 3, 4]
HORIZONS = [3, 6, 12, 24]                          # hours ahead
RANDOM_STATE = 42
N_SPLITS = 5                                       # GroupKFold splits (stations)


In [2]:
# ============================================
# === Cell 2: Helper Functions (metrics)   ===
# ============================================

def expected_calibration_error(y_true, y_prob, n_bins=15):
    """
    Expected Calibration Error (ECE) with equal-width bins.

    y_true: array-like of 0/1
    y_prob: array-like of [0,1] probabilities
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_true = y_true[mask].mean()
        bin_pred = y_prob[mask].mean()
        ece += abs(bin_pred - bin_true) * (mask.sum() / n)

    return float(ece)

In [3]:
# ============================================
# === Cell 3: Load Data & Basic Features  ===
# ============================================

def load_and_basic_features(csv_path: Path) -> pd.DataFrame:
    """
    Load CIMIS CSV and construct:
    - datetime
    - cool-season filter
    - basic time features (day-of-year, hour-of-day sin/cos)
    """
    print(f"[INFO] Loading data from: {csv_path}")
    df = pd.read_csv(csv_path)

    # Build datetime
    df["date"] = pd.to_datetime(df["Date"])
    # Hour (PST) is like 100, 200, ..., 2300 -> convert to integer hour 1..23
    df["hour"] = (df["Hour (PST)"] // 100).astype(int)
    df["datetime"] = df["date"] + pd.to_timedelta(df["hour"], unit="h")

    # Cool season filter
    df["month"] = df["datetime"].dt.month
    df = df[df["month"].isin(COOL_MONTHS)].copy()

    # Time features
    df["doy"] = df["datetime"].dt.dayofyear
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

    df = df.sort_values(["Stn Id", "datetime"]).reset_index(drop=True)

    print(f"[INFO] Rows after cool-season filter: {df.shape[0]}")
    return df

# Actually load the data now:
df_raw = load_and_basic_features(DATA_PATH)
df_raw.head()


[INFO] Loading data from: cimis_all_stations_clean.csv
[INFO] Rows after cool-season filter: 1375488


,Stn Id,Stn Name,CIMIS Region,Date,Hour (PST),Jul,ETo (mm),qc,Precip (mm),qc.1,...,qc.8,Soil Temp (C),qc.9,date,hour,datetime,month,doy,hour_sin,hour_cos
0,2,FivePoints,San Joaquin Valley,9/30/2010,2400,273,0.04,Y,0.0,,...,,22.3,,2010-09-30,24,2010-10-01 00:00:00,10,274,-2.449294e-16,1.000000
1,2,FivePoints,San Joaquin Valley,10/1/2010,100,274,0.04,Y,0.0,,...,,22.3,,2010-10-01,1,2010-10-01 01:00:00,10,274,2.588190e-01,0.965926
2,2,FivePoints,San Joaquin Valley,10/1/2010,200,274,0.04,Y,0.0,,...,,22.2,,2010-10-01,2,2010-10-01 02:00:00,10,274,5.000000e-01,0.866025
3,2,FivePoints,San Joaquin Valley,10/1/2010,300,274,0.02,Y,0.0,,...,,22.1,,2010-10-01,3,2010-10-01 03:00:00,10,274,7.071068e-01,0.707107
4,2,FivePoints,San Joaquin Valley,10/1/2010,400,274,0.02,Y,0.0,,...,,22.1,,2010-10-01,4,2010-10-01 04:00:00,10,274,8.660254e-01,0.500000


In [4]:
# ============================================
# === Cell 4: Lag Features & Targets      ===
# ============================================

def add_lag_features(df: pd.DataFrame, max_lag: int = 3) -> pd.DataFrame:
    """
    Add lag features for key columns per station to capture recent history.
    """
    df = df.sort_values(["Stn Id", "datetime"]).copy()

    lag_source_cols = [
        "Air Temp (C)",
        "Rel Hum (%)",
        "Dew Point (C)",
        "Soil Temp (C)",
    ]

    print(f"[INFO] Adding lag features up to {max_lag} hours for: {lag_source_cols}")
    for col in lag_source_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column for lags: {col}")
        for lag in range(1, max_lag + 1):
            df[f"{col}_lag{lag}"] = df.groupby("Stn Id")[col].shift(lag)

    return df


def add_targets_for_horizon(df: pd.DataFrame, horizon_h: int) -> pd.DataFrame:
    """
    For horizon H, add:
    - temp_t+Hh  : future air temperature at t+H
    - frost_Hh   : 1 if temp_t+Hh < 0°C, else 0
    """
    df = df.sort_values(["Stn Id", "datetime"]).copy()

    if "Air Temp (C)" not in df.columns:
        raise ValueError("Column 'Air Temp (C)' required for targets.")

    future_temp = df.groupby("Stn Id")["Air Temp (C)"].shift(-horizon_h)
    df[f"temp_t+{horizon_h}h"] = future_temp
    df[f"frost_{horizon_h}h"] = (future_temp < 0).astype(int)

    return df


def get_feature_columns(df: pd.DataFrame):
    """
    List of feature columns used by the model.
    """
    base_features = [
        "Air Temp (C)",
        "Rel Hum (%)",
        "Dew Point (C)",
        "Wind Speed (m/s)",
        "Wind Dir (0-360)",
        "Soil Temp (C)",
        "Sol Rad (W/sq.m)",
        "Vap Pres (kPa)",
        "Precip (mm)",
        "ETo (mm)",
        "Jul",
        "hour_sin",
        "hour_cos",
        "doy",
    ]

    for col in base_features:
        if col not in df.columns:
            raise ValueError(f"Required base feature missing: {col}")

    lag_cols = [c for c in df.columns if "lag" in c]

    feature_cols = base_features + lag_cols
    print(f"[INFO] Total feature columns: {len(feature_cols)}")
    return feature_cols


def build_training_matrix_for_horizon(
    df: pd.DataFrame,
    horizon_h: int,
    feature_cols,
):
    """
    Build X, y_cls, y_reg, groups for a specific horizon.
    Drops rows with any NaNs in features/targets.
    """
    target_cls = f"frost_{horizon_h}h"
    target_reg = f"temp_t+{horizon_h}h"

    needed = feature_cols + [target_cls, target_reg, "Stn Id"]
    df_sub = df[needed].dropna().copy()

    X = df_sub[feature_cols].astype(float)
    y_cls = df_sub[target_cls].astype(int)
    y_reg = df_sub[target_reg].astype(float)
    groups = df_sub["Stn Id"].values

    print(
        f"[INFO][H={horizon_h}h] Training rows: {X.shape[0]}, "
        f"frost rate={y_cls.mean():.4f}"
    )
    return X, y_cls, y_reg, groups


# Build lags once:
df_fe = add_lag_features(df_raw, max_lag=3)

# Just to inspect feature columns:
feature_cols = get_feature_columns(df_fe)
feature_cols[:10], " ...", feature_cols[-5:]


[INFO] Adding lag features up to 3 hours for: ['Air Temp (C)', 'Rel Hum (%)', 'Dew Point (C)', 'Soil Temp (C)']
[INFO] Total feature columns: 26


(['Air Temp (C)',
  'Rel Hum (%)',
  'Dew Point (C)',
  'Wind Speed (m/s)',
  'Wind Dir (0-360)',
  'Soil Temp (C)',
  'Sol Rad (W/sq.m)',
  'Vap Pres (kPa)',
  'Precip (mm)',
  'ETo (mm)'],
 ' ...',
 ['Dew Point (C)_lag2',
  'Dew Point (C)_lag3',
  'Soil Temp (C)_lag1',
  'Soil Temp (C)_lag2',
  'Soil Temp (C)_lag3'])

In [7]:
# ============================================
# === Cell 5: GroupKFold Training & CV    ===
# ============================================

def train_and_evaluate_horizon(
    df_fe: pd.DataFrame,
    horizon_h: int,
    feature_cols,
    n_splits: int = N_SPLITS,
):
    """
    For one horizon H:
    - add targets
    - build training matrix
    - run GroupKFold CV by station
    - train LightGBM classifier (frost) + regressor (temp)
    - return:
        overall_metrics (dict)
        final_clf, final_reg (trained on ALL data)
    """
    print("\n" + "=" * 70)
    print(f"[INFO] Training for horizon H={horizon_h} hours")
    print("=" * 70)

    df_h = add_targets_for_horizon(df_fe, horizon_h)
    X, y_cls, y_reg, groups = build_training_matrix_for_horizon(
        df_h, horizon_h, feature_cols
    )

    if len(np.unique(y_cls)) < 2:
        raise RuntimeError(f"H={horizon_h}h: only one class present, cannot train.")

    gkf = GroupKFold(n_splits=n_splits)
    n = len(X)

    # Out-of-fold predictions for evaluation
    oof_prob = np.zeros(n, dtype=float)
    oof_temp = np.zeros(n, dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y_cls, groups), start=1):
        print(f"\n[INFO] Fold {fold}/{n_splits} (H={horizon_h}h)")

        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr_cls, y_va_cls = y_cls.iloc[tr_idx], y_cls.iloc[va_idx]
        y_tr_reg, y_va_reg = y_reg.iloc[tr_idx], y_reg.iloc[va_idx]

        # LightGBM classifier: balanced, regularized to avoid overfitting
        clf = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=63,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_samples=50,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            class_weight="balanced",
        )
        clf.fit(X_tr, y_tr_cls)
        prob_va = clf.predict_proba(X_va)[:, 1]
        oof_prob[va_idx] = prob_va

        # LightGBM regressor for temperature
        reg = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=63,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_samples=50,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        reg.fit(X_tr, y_tr_reg)
        temp_va = reg.predict(X_va)
        oof_temp[va_idx] = temp_va

        fold_brier = brier_score_loss(y_va_cls, prob_va)
        fold_roc = roc_auc_score(y_va_cls, prob_va)
        print(f"[INFO] Fold {fold}: Brier={fold_brier:.4f}, ROC-AUC={fold_roc:.4f}")

    # Overall metrics
    metrics = {}
    metrics["brier"] = brier_score_loss(y_cls, oof_prob)
    metrics["roc_auc"] = roc_auc_score(y_cls, oof_prob)
    metrics["pr_auc"] = average_precision_score(y_cls, oof_prob)
    metrics["ece"] = expected_calibration_error(y_cls, oof_prob, n_bins=15)
    metrics["rmse_temp"] = mean_squared_error(y_reg, oof_temp) ** 0.5


    print("\n[INFO] Overall metrics (H={}h)".format(horizon_h))
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    # Train final models on ALL data for deployment
    final_clf = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_samples=50,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )
    final_clf.fit(X, y_cls)

    final_reg = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_samples=50,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    final_reg.fit(X, y_reg)

    return metrics, final_clf, final_reg

In [8]:
# ============================================
# === Cell 6: Run Training for All H      ===
# ============================================

# Dictionary to store final models for each horizon
models_by_horizon = {}
metrics_rows = []

for H in HORIZONS:
    metrics_H, clf_H, reg_H = train_and_evaluate_horizon(df_fe, H, feature_cols)
    metrics_row = {"horizon_h": H}
    metrics_row.update(metrics_H)
    metrics_rows.append(metrics_row)
    models_by_horizon[H] = {
        "classifier": clf_H,
        "regressor": reg_H,
        "feature_cols": feature_cols,
    }

metrics_df = pd.DataFrame(metrics_rows)
metrics_df


[INFO] Training for horizon H=3 hours
[INFO][H=3h] Training rows: 1314198, frost rate=0.0152

[INFO] Fold 1/5 (H=3h)
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 16764, number of negative: 1010769
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003639 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4676
[LightGBM] [Info] Number of data points in the train set: 1027533, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

,horizon_h,brier,roc_auc,pr_auc,ece,rmse_temp
0,3,0.016305,0.996213,0.845692,0.021521,1.454409
1,6,0.023808,0.993015,0.744782,0.031967,2.043618
2,12,0.034686,0.985884,0.586292,0.049640,2.512837
3,24,0.043613,0.981912,0.566724,0.066212,2.433322


In [9]:
# ============================================
# === Cell 7: Prediction Function         ===
# ============================================

def build_features_for_prediction(df_raw_like: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the SAME feature logic to a new CIMIS-style dataframe:
    - datetime, cool-season filter, time features, lags
    """
    df = df_raw_like.copy()

    df["date"] = pd.to_datetime(df["Date"])
    df["hour"] = (df["Hour (PST)"] // 100).astype(int)
    df["datetime"] = df["date"] + pd.to_timedelta(df["hour"], unit="h")

    df["month"] = df["datetime"].dt.month
    df = df[df["month"].isin(COOL_MONTHS)].copy()

    df["doy"] = df["datetime"].dt.dayofyear
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

    df = df.sort_values(["Stn Id", "datetime"]).reset_index(drop=True)

    # Add lags (same as before)
    df = add_lag_features(df, max_lag=3)

    return df


def build_prediction_matrix(df_fe_pred: pd.DataFrame, feature_cols):
    """
    Prepare X and an ID index for merging predictions back.
    Rows missing lags will be dropped.
    """
    df = df_fe_pred.copy()
    df["row_id"] = df.index

    # ensure all feature cols exist
    for col in feature_cols:
        if col not in df.columns:
            raise ValueError(f"Feature column missing in prediction data: {col}")

    cols = feature_cols + ["Stn Id", "datetime", "row_id"]
    df_sub = df[cols].dropna().copy()

    X = df_sub[feature_cols].astype(float)
    return df_sub, X


def predict_for_dataset(
    csv_path: Path,
    models_by_horizon: dict,
    feature_cols,
) -> pd.DataFrame:
    """
    Load a CIMIS-style CSV and produce predictions for each horizon:
    - frost_prob_{H}h
    - temp_pred_{H}h
    - target_time_{H}h
    """
    print(f"[INFO] Loading data for prediction from: {csv_path}")
    df_new_raw = pd.read_csv(csv_path)
    df_new_fe = build_features_for_prediction(df_new_raw)

    base = df_new_fe.copy()
    base["row_id"] = base.index

    preds_list = []

    for H, bundle in models_by_horizon.items():
        print(f"[INFO] Predicting for horizon H={H}h")

        df_sub, X_new = build_prediction_matrix(df_new_fe, bundle["feature_cols"])
        clf = bundle["classifier"]
        reg = bundle["regressor"]

        frost_prob = clf.predict_proba(X_new)[:, 1]
        temp_pred = reg.predict(X_new)
        target_time = df_sub["datetime"] + pd.to_timedelta(H, unit="h")

        tmp = pd.DataFrame(
            {
                "row_id": df_sub["row_id"].values,
                f"frost_prob_{H}h": frost_prob,
                f"temp_pred_{H}h": temp_pred,
                f"target_time_{H}h": target_time,
            }
        )
        preds_list.append(tmp)

    # Merge predictions back
    preds_all = base[["row_id", "Stn Id", "datetime"]].copy()
    for tmp in preds_list:
        preds_all = preds_all.merge(tmp, on="row_id", how="left")

    preds_all = preds_all.sort_values(["Stn Id", "datetime"]).reset_index(drop=True)
    return preds_all


In [ ]:
# ============================================
# === Cell 8: Example Prediction          ===
# ============================================

# Example: use the SAME historical CSV just to see predictions structure.
# In practice, you would pass a NEW CSV path here.
preds_example = predict_for_dataset(DATA_PATH, models_by_horizon, feature_cols)

preds_example.head()